# Week 5 — 手写单头 Self-Attention

**目标**：从零实现 scaled dot-product attention 和单头 self-attention，在一个玩具任务上训练并可视化 attention 权重。

**产出**：
- `ScaledDotProductAttention` / `SingleHeadSelfAttention` 两个 `nn.Module`
- 在 toy 任务上的 loss 曲线 + attention 热力图
- sqrt(d_k) 缩放消融实验


In [ ]:
# ── Bootstrap (Colab + local) ─────────────────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')


## 1. 为什么有 Attention？Q/K/V 的直觉

把 self-attention 看作一次**软字典查询**：

- **Query (Q)**：我想找什么？（当前位置的问题）
- **Key (K)**：每个位置的索引标签（"我是什么"）
- **Value (V)**：每个位置实际携带的信息

计算：用 `Q · K^T` 打分 → `softmax` 转成权重 → 对 `V` 做加权平均。

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

**为什么除以 $\sqrt{d_k}$**？

假设 $Q, K$ 各分量独立同分布、均值 0 方差 1，那么 $Q\cdot K$ 的方差是 $d_k$，标准差是 $\sqrt{d_k}$。如果不缩放，$d_k$ 大的时候 softmax 输入方差大 → 输出进入饱和区 → 梯度就几乎为 0。

**Self-attention vs RNN**：

| 维度 | RNN / LSTM | Self-Attention |
|----|----|----|
| 串行 / 并行 | 顺序展开，时间上串行 | 所有位置并行计算 |
| 长依赖 | 信息顺着隐状态传递，路径 O(L) | 任意两位置直接连接，O(1) |
| 计算复杂度 | O(L·d²) | O(L²·d) （L 小时反而便宜）|
| 可解释性 | 隐藏状态黑盒 | attention 权重直接可视化 |


## 2. Toy 任务：让 attention 有用武之地

**输入**：长度 8 的整数序列，每个 token 在 `{0,1,...,15}` 中。

**标签**：前缀和 (prefix sum) 在某个位置首次跨过 50，标签为 1；否则为 0。

**为什么这个任务能冸显出 attention 的优势**：
- 需要全局信息（判断跨阈值要求和所有已计入的 token）。
- bag-of-tokens 的线性模型没有顺序感，看不到"前缀"的概念。
- 小 MLP 看到拼接后的位置信息，但没有情境聚合的能力。

先造数据集，再跟 MLP baseline 比一比。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB, SEQ_LEN, THRESH = 16, 8, 50

def make_dataset(n):
    # tokens in [0, 16), length 8
    x = torch.randint(0, VOCAB, (n, SEQ_LEN))
    prefix = x.cumsum(dim=1)
    # label = 1 iff prefix sum ever crosses THRESH
    y = (prefix.max(dim=1).values > THRESH).long()
    return x, y

torch.manual_seed(42)
X_tr, y_tr = make_dataset(4096)
X_va, y_va = make_dataset(1024)
print('train shape', X_tr.shape, 'pos rate', y_tr.float().mean().item())
print('valid shape', X_va.shape, 'pos rate', y_va.float().mean().item())
print('example tokens :', X_tr[0].tolist())
print('example prefix :', X_tr[0].cumsum(0).tolist())
print('example label  :', y_tr[0].item())


### 2.1 MLP baseline：看看不用 attention 能多好

拼接 embedding 后直接过两层 MLP。预期能学一点但在临界样本上吃力。


In [ ]:
class MLPBaseline(nn.Module):
    def __init__(self, vocab=VOCAB, d=32, seq_len=SEQ_LEN, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.net = nn.Sequential(
            nn.Linear(d * seq_len, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        h = self.emb(x).flatten(1)
        return self.net(h).squeeze(-1)


def train_clf(model, X_tr, y_tr, X_va, y_va, steps=2000, bs=128, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    X_tr_d, y_tr_d = X_tr.to(device), y_tr.float().to(device)
    X_va_d, y_va_d = X_va.to(device), y_va.float().to(device)
    for step in range(steps):
        idx = torch.randint(0, len(X_tr_d), (bs,), device=device)
        logit = model(X_tr_d[idx])
        loss = F.binary_cross_entropy_with_logits(logit, y_tr_d[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    with torch.no_grad():
        acc = ((model(X_va_d) > 0) == (y_va_d > 0.5)).float().mean().item()
    return losses, acc

torch.manual_seed(42)
mlp = MLPBaseline()
mlp_losses, mlp_acc = train_clf(mlp, X_tr, y_tr, X_va, y_va)
print(f'MLP baseline valid acc = {mlp_acc:.3f}')


## 3. 手写 `ScaledDotProductAttention`

输入 `Q, K, V` 形状 `(B, L, d)`，返回：
- `out`：`(B, L, d)`
- `attn`：`(B, L, L)` 权重矩阵（后续画图用）

只用 `torch.matmul` + `F.softmax`，不调 `nn.MultiheadAttention`。


In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

    shapes:
        Q, K, V: (B, L, d_k)
        returns: out (B, L, d_v), attn (B, L, L)
    """
    def __init__(self, scaled: bool = True):
        super().__init__()
        self.scaled = scaled

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, L, L)
        if self.scaled:
            scores = scores / (d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        return out, attn


# sanity check
sdpa = ScaledDotProductAttention()
B, L, d = 2, 8, 32
q = torch.randn(B, L, d); k = torch.randn(B, L, d); v = torch.randn(B, L, d)
out, attn = sdpa(q, k, v)
print('out :', out.shape, '  attn :', attn.shape)
print('attn row sums (should be ~1):', attn.sum(dim=-1)[0, :3].tolist())


## 4. 手写 `SingleHeadSelfAttention`

封装成：输入 `x: (B, L, d)` → 学习投影 `W_q, W_k, W_v` → 算 attention → 输出投影 `W_o`。

强调：
- Q/K/V 都来自同一个 `x`（这就是 "self" attention 的含义）。
- 最后的 `W_o` 在单头下也保留，为了和下周 multi-head 结构对齐。


In [ ]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, scaled: bool = True):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.attn_fn = ScaledDotProductAttention(scaled=scaled)
        self.last_attn = None  # store for viz

    def forward(self, x, mask=None):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        out, attn = self.attn_fn(Q, K, V, mask=mask)
        self.last_attn = attn.detach()
        return self.W_o(out)

# sanity check
sha = SingleHeadSelfAttention(d_model=32)
y = sha(torch.randn(B, L, 32))
print('output shape :', y.shape)
print('stored attn  :', sha.last_attn.shape)


## 5. 小分类器：Embedding → SelfAttention → mean-pool → Linear

简单到不能再简单。后面的 multi-head / PE / EncoderBlock 都是在这个骨架上加点东西。


In [ ]:
class ToyAttnClassifier(nn.Module):
    def __init__(self, vocab=VOCAB, d=32, scaled=True):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.attn = SingleHeadSelfAttention(d, scaled=scaled)
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        h = self.emb(x)              # (B, L, d)
        h = self.attn(h)             # (B, L, d)
        h = h.mean(dim=1)            # (B, d)  mean-pool over positions
        return self.head(h).squeeze(-1)

    def attn_weights(self, x):
        _ = self.forward(x)
        return self.attn.last_attn    # (B, L, L)

torch.manual_seed(42)
model = ToyAttnClassifier()
print('params :', sum(p.numel() for p in model.parameters()))
attn_losses, attn_acc = train_clf(model, X_tr, y_tr, X_va, y_va, steps=2000, lr=1e-3)
print(f'single-head attention valid acc = {attn_acc:.3f}  (MLP was {mlp_acc:.3f})')


## 6. 训练曲线

MLP vs single-head attention。预期后者收敛更快且最终 loss 更低。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def smooth(xs, k=50):
    xs = np.asarray(xs)
    if len(xs) < k: return xs
    c = np.convolve(xs, np.ones(k)/k, mode='valid')
    return c

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(smooth(mlp_losses), label=f'MLP (acc={mlp_acc:.3f})')
ax.plot(smooth(attn_losses), label=f'SingleHead Attn (acc={attn_acc:.3f})')
ax.set_xlabel('step (smoothed)'); ax.set_ylabel('loss'); ax.set_title('Toy task: loss curves')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7. 可视化 attention 权重

挑两个样本：一个 "刚跨过阈值" 的 positive，一个 negative。看模型在查询每一个位置时，它所关注的是哪几个位置。

一个好的学习迹象：高值 token（接近 15）的列平均权重应当比平均高——因为它们对"跨过 50"贡献更大。


In [ ]:
model.eval()
with torch.no_grad():
    pos_idx = (y_va == 1).nonzero(as_tuple=True)[0][0].item()
    neg_idx = (y_va == 0).nonzero(as_tuple=True)[0][0].item()
    samples = X_va[[pos_idx, neg_idx]].to(device)
    W = model.attn_weights(samples).cpu().numpy()  # (2, L, L)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, idx, W_i, title in zip(
    axes, [pos_idx, neg_idx], W,
    ['label=1 (prefix crossed 50)', 'label=0 (never crossed)']
):
    toks = X_va[idx].tolist()
    im = ax.imshow(W_i, cmap='viridis', aspect='auto', vmin=0, vmax=W_i.max())
    ax.set_xticks(range(SEQ_LEN)); ax.set_xticklabels(toks)
    ax.set_yticks(range(SEQ_LEN)); ax.set_yticklabels(toks)
    ax.set_xlabel('KEY position (token value)')
    ax.set_ylabel('QUERY position (token value)')
    ax.set_title(f'{title}\ntokens={toks}')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


## 8. 消融：移除 sqrt(d_k) 缩放会怎样？

理论预期：不缩放时当 `d=32`，QK^T 的方差 ≈ 32，softmax 容易饱和，梯度变小——表现为 **训练初期 loss 下降更慢、整体不稳定**。


In [ ]:
torch.manual_seed(42)
model_noscale = ToyAttnClassifier(scaled=False)
noscale_losses, noscale_acc = train_clf(model_noscale, X_tr, y_tr, X_va, y_va, steps=2000, lr=1e-3)
print(f'no-sqrt scaling valid acc = {noscale_acc:.3f}  (scaled was {attn_acc:.3f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(smooth(attn_losses),    label=f'scaled   (acc={attn_acc:.3f})')
ax.plot(smooth(noscale_losses), label=f'unscaled (acc={noscale_acc:.3f})')
ax.set_xlabel('step (smoothed)'); ax.set_ylabel('loss')
ax.set_title('Ablation: with / without sqrt(d_k) scaling')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 9. 复盘三问（内联答案）

**Q1. Self-attention 相比 RNN 有哪些优势？**

1. **并行**：所有位置的 Q/K/V 投影和 attention 计算均能一次序列维度矩阵乘法完成，而 RNN 必须顺时间展开。
2. **O(1) 路径长度**：任意两个 token 之间只隔一个 attention 步，梯度不会随序列长度指数衰减；RNN 是 O(L)。
3. **可解释**：attention 权重可以直接画出来（就像上面的热力图），RNN 的隐藏状态很难直接解读。

**Q2. 为什么除以 sqrt(d_k)？**

假设 Q 和 K 的各分量独立同分布、均值 0 方差 1：

$$\mathrm{Var}(Q \cdot K) = \sum_{i=1}^{d_k} \mathrm{Var}(Q_i K_i) = d_k$$

标准差为 $\sqrt{d_k}$。如果不缩放，softmax 的输入幅度随 $d_k$ 增大，最大值被放大→softmax 进入饱和区（输出接近 one-hot）→输出对 score 的梯度几乎为 0→训练卡住。除以 $\sqrt{d_k}$ 正好让 score 的方差回到 1，softmax 基本不饱和。

**Q3. 实验观察：移除缩放后怎么样？**

看上面的消融曲线：`scaled=False` 下初期 loss 下降明显更慢、更振荡，最终准确率也低一截。这与“softmax 饱和 → 梯度消失”的理论预期一致。

---

下周（Week 6）：把单头拆成 multi-head，加 positional encoding 和 feed-forward，组装成完整的 Encoder Block。
